In [1]:
!pip install transformers[torch] torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
import torch
from torch.utils.data import Dataset
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjit

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
checkpoint_path = "/content/bias_media_eai_folder"  # Choose the best checkpoint based on performance
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

In [8]:
fine_tune_df = pd.read_csv("/content/bias_prediction_articles.csv")  # Replace with your data
fine_tune_df['text'] = fine_tune_df['title'].fillna('') + " [SEP] " + fine_tune_df['content'].fillna('')

In [9]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [10]:
fine_tune_dataset = NewsDataset(
    fine_tune_df['text'].tolist(),
    fine_tune_df['bias'].tolist(),
    tokenizer,
    512
)

In [11]:
fine_tuning_args = TrainingArguments(
    output_dir="./fine_tuned_media_bias_model",
    num_train_epochs=4,                     # Fewer epochs for fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,                     # Lower learning rate (default is 5e-5)
    weight_decay=0.01,                      # Regularization to prevent overfitting
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True                               # Use mixed precision if you have compatible GPU
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [13]:
from sklearn.model_selection import train_test_split
fine_tune_texts = fine_tune_df['text'].tolist()
fine_tune_labels = fine_tune_df['bias'].tolist()

train_texts, eval_texts, train_labels, eval_labels = train_test_split(
    fine_tune_texts, fine_tune_labels, test_size=0.2, random_state=42
)

# Create datasets
train_dataset = NewsDataset(train_texts, train_labels, tokenizer, 512)
eval_dataset = NewsDataset(eval_texts, eval_labels, tokenizer, 512)

In [14]:
trainer = Trainer(
    model=model,
    args=fine_tuning_args,
    train_dataset=fine_tune_dataset,
    eval_dataset=eval_dataset
)

In [15]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: moham09 (moham09-pfw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.296500,0.162260
2,0.212200,0.112536
3,0.150700,0.057313
4,0.068100,0.031202


TrainOutput(global_step=9392, training_loss=0.1829061014688807, metrics={'train_runtime': 1547.7712, 'train_samples_per_second': 97.053, 'train_steps_per_second': 6.068, 'total_flos': 3.952384515742925e+16, 'train_loss': 0.1829061014688807, 'epoch': 4.0})

In [16]:
!zip -r /content/fine_tuned_media_bias_model.zip /content/fine_tuned_media_bias_model
from google.colab import files
files.download('/content/fine_tuned_media_bias_model.zip')

  adding: content/fine_tuned_media_bias_model/ (stored 0%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/ (stored 0%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/scheduler.pt (deflated 55%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/config.json (deflated 51%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/optimizer.pt (deflated 11%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/scaler.pt (deflated 60%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/trainer_state.json (deflated 70%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/training_args.bin (deflated 51%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/model.safetensors (deflated 7%)
  adding: content/fine_tuned_media_bias_model/checkpoint-7044/rng_state.pth (deflated 25%)
  adding: content/fine_tuned_media_bias_model/checkpoint-9392/ (stored 0%)
  adding: content/fine_tuned_media_bias_model/checkpoint-93

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>